# Lab 5 — FeatureTools Auto Feature Engineering

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Build a FeatureTools **EntitySet** from the loans table.
2. Run **Deep Feature Synthesis (DFS)** with `max_depth=1`.
3. Inspect the engineered **feature matrix** shape and column names.
4. Rank auto-generated features by **|correlation|** with `default`.

> **Checkpoints:** shape **(1000, 6)** · **6** engineered features · top corr = **int_rate** (~0.21)



## Deep Feature Synthesis (DFS) in one slide

FeatureTools automates feature engineering by applying **primitives** (sum, mean, count, etc.) across entities in an **EntitySet**.

| Concept | This lab |
|---------|----------|
| **EntitySet** | One table: `loans` indexed by `loan_id` |
| **DFS** | `ft.dfs(..., max_depth=1)` — one hop of transformations |
| **Output** | `feature_matrix` (1000 rows) + `feature_defs` (column recipes) |

With a single table and `max_depth=1`, DFS mainly creates **aggregations/transforms** of numeric columns — a starting point before hand-crafted features (Day 3) or model training.

---

## 1. Load loans and select DFS columns

In [ ]:
from pathlib import Path

import featuretools as ft
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-04":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
dfs_cols = [
    "loan_id",
    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "installment",
    "default",
]

raw = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
raw["default"] = raw["loan_status"].isin(DEFAULT_STATUSES).astype(int)
df = raw[dfs_cols].copy()
df["loan_id"] = df["loan_id"].astype(str)

print(f"input columns: {len(df.columns)}")
print(f"rows: {len(df)}")
display(df.head(3))

`loan_id` becomes the **index** for the entity. `default` stays in the frame so DFS can include it in the output matrix for correlation analysis.

---

## 2. Build the EntitySet

In [ ]:
es = ft.EntitySet(id="lending")
es = es.add_dataframe(
    dataframe_name="loans",
    dataframe=df,
    index="loan_id",
)

print(es)

---

## 3. Run Deep Feature Synthesis

In [ ]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="loans",
    max_depth=1,
    verbose=False,
)

print("Lab 5 — FeatureTools auto FE")
print(f"engineered features: {len(feature_defs)}")
print(f"feature matrix shape: {feature_matrix.shape}")
display(feature_matrix.head(3))

---

## 4. Inspect feature definitions

In [ ]:
def_names = [fd.get_name() for fd in feature_defs]
defs_df = pd.DataFrame({"feature": def_names})
display(defs_df)

Each `feature_def` records the **primitive** and input columns — useful for reproducing or auditing auto-generated features in production.

---

## 5. Correlation with default

In [ ]:
numeric_cols = [
    c for c in feature_matrix.columns
    if c != "default" and str(feature_matrix[c].dtype) != "category"
]
top_corr = (
    feature_matrix[numeric_cols + ["default"]]
    .corr(numeric_only=True)["default"]
    .drop("default", errors="ignore")
    .abs()
    .sort_values(ascending=False)
)

print("top |corr| with default (first 3):")
for name, val in top_corr.head(3).items():
    print(f"  {name}: {val:.4f}")

display(top_corr.head(5).to_frame("abs_corr").round(4))

**int_rate** leads — consistent with Day 3 logistic regression where `int_rate` had the strongest positive coefficient.

---

## 6. Compare to hand-picked Day 3 features

In [ ]:
hand_picked = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
hand_corr = (
    raw[hand_picked + ["default"]]
    .corr(numeric_only=True)["default"]
    .drop("default")
    .abs()
    .sort_values(ascending=False)
)

compare = pd.DataFrame({
    "source": ["DFS top", "Hand-picked top"],
    "feature": [top_corr.index[0], hand_corr.index[0]],
    "abs_corr": [top_corr.iloc[0], hand_corr.iloc[0]],
})
display(compare.round(4))

DFS surfaces the same signal domain experts chose — but on richer multi-table data it can discover interactions you'd miss manually.

---

## 7. Checkpoint summary

In [ ]:
assert len(df.columns) == 7
assert len(feature_defs) == 6
assert feature_matrix.shape == (1000, 6)
assert top_corr.index[0] == "int_rate"
assert abs(top_corr.iloc[0] - 0.2084) < 0.02
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. What new features might appear with `max_depth=2` or multiple related tables?
2. Why filter out categorical columns before correlation ranking?
3. Would you feed the DFS matrix directly into KNN without reviewing primitives?

**Previous:** [Lab 4 — FastAPI scoring API](lab04_fastapi_scoring_api.ipynb)  
**Next:** [Lab 6 — MLflow experiment log](lab06_mlflow_experiment_log.ipynb)